# 13C NMR Chemical Shift Prediction — Post-Training Analytics

In [ ]:
import json, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import pearsonr

plt.rcParams.update({
    "figure.dpi": 130,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

MODES  = ["topo", "xyz", "distances"]
COLORS = {"topo": "#4C72B0", "xyz": "#DD8452", "distances": "#55A868"}
LABELS = {"topo": "Topology only", "xyz": "Topology + 3-D coords", "distances": "Topology + bond length"}
PPM_BUCKETS = [(0, 50), (50, 100), (100, 150), (150, 200), (200, 300)]

## Load results

In [ ]:
results = {}
for mode in MODES:
    path = f"results_{mode}.json"
    if os.path.exists(path):
        results[mode] = json.load(open(path))
        cfg = results[mode]["config"]
        print(f"[{mode}]  test_mae={results[mode]['test_mae']:.2f} ppm  "
              f"epochs={cfg['epochs']}  lr={cfg['lr']}  "
              f"hidden={cfg['hidden_dim']}  layers={cfg['num_layers']}  drop={cfg['dropout']}")
    else:
        print(f"  (missing {path} — skipping)")

loaded = list(results.keys())
print(f"\nLoaded: {loaded}")

## Training & Validation Loss Curves

In [ ]:
n = len(loaded)
if n == 0:
    print("No results loaded.")
else:
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4), sharey=False)
    if n == 1:
        axes = [axes]

    for ax, mode in zip(axes, loaded):
        r = results[mode]
        train_maes = r["train_maes"]
        val_maes   = r["val_maes"]
        epochs     = range(1, len(train_maes) + 1)
        best_ep    = int(np.argmin(val_maes)) + 1
        color      = COLORS[mode]

        ax.plot(epochs, train_maes, color=color, lw=2, label="Train MAE")
        ax.plot(epochs, val_maes,   color=color, lw=2, ls="--", label="Val MAE")
        ax.axvline(best_ep, color="grey", ls=":", lw=1.2)
        ax.text(best_ep + 0.3, ax.get_ylim()[1] * 0.98,
                f"best ep {best_ep}\n{min(val_maes):.2f} ppm",
                va="top", fontsize=8, color="grey")
        ax.set_title(LABELS[mode], fontweight="bold")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("MAE (ppm)")
        ax.legend(fontsize=9)

    fig.suptitle("Training & Validation Loss Curves", fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig("fig_loss_curves.png", bbox_inches="tight")
    plt.show()

## Model Comparison: Train / Val / Test MAE

In [ ]:
if not loaded:
    print("No results loaded.")
else:
    bar_w   = 0.25
    x       = np.arange(len(loaded))
    metrics = ["Train (final)", "Val (best)", "Test"]
    bar_colors = ["#6baed6", "#fd8d3c", "#74c476"]

    vals = {
        "Train (final)": [results[m]["train_maes"][-1] for m in loaded],
        "Val (best)":    [results[m]["best_val_mae"]   for m in loaded],
        "Test":          [results[m]["test_mae"]        for m in loaded],
    }

    fig, ax = plt.subplots(figsize=(max(5, 3 * len(loaded)), 4.5))
    for i, (label, color) in enumerate(zip(metrics, bar_colors)):
        bars = ax.bar(x + (i - 1) * bar_w, vals[label], bar_w,
                      label=label, color=color, edgecolor="white", linewidth=0.5)
        for bar, v in zip(bars, vals[label]):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.15,
                    f"{v:.1f}", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels([LABELS[m] for m in loaded])
    ax.set_ylabel("MAE (ppm)")
    ax.set_title("Model Comparison — MAE by Split", fontweight="bold")
    ax.legend()
    plt.tight_layout()
    plt.savefig("fig_model_comparison.png", bbox_inches="tight")
    plt.show()

## Predicted vs Actual Scatter

In [ ]:
if not loaded:
    print("No results loaded.")
else:
    n   = len(loaded)
    fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 5), sharey=False)
    if n == 1:
        axes = [axes]

    for ax, mode in zip(axes, loaded):
        pred = np.array(results[mode]["test_pred"])
        true = np.array(results[mode]["test_true"])
        mae  = np.mean(np.abs(pred - true))
        r, _ = pearsonr(pred, true)
        ss_res = np.sum((true - pred) ** 2)
        ss_tot = np.sum((true - np.mean(true)) ** 2)
        r2 = 1 - ss_res / ss_tot

        lim = (min(true.min(), pred.min()) - 5, max(true.max(), pred.max()) + 5)
        ax.hexbin(true, pred, gridsize=60, cmap="Blues", mincnt=1, linewidths=0.2)
        ax.plot(lim, lim, "r-", lw=1.5, label="y = x")
        ax.set_xlim(lim); ax.set_ylim(lim)
        ax.set_xlabel("True shift (ppm)")
        ax.set_ylabel("Predicted shift (ppm)")
        ax.set_title(LABELS[mode], fontweight="bold")
        ax.text(0.04, 0.96,
                f"MAE = {mae:.2f} ppm\nr = {r:.3f}\nR² = {r2:.3f}",
                transform=ax.transAxes, va="top", fontsize=9,
                bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.7))
        ax.legend(fontsize=9)

    fig.suptitle("Predicted vs Actual Chemical Shift (Test Set)", fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig("fig_scatter.png", bbox_inches="tight")
    plt.show()

## MAE by ppm Range

In [ ]:
if not loaded:
    print("No results loaded.")
else:
    bucket_labels = [f"{lo}–{hi}" for lo, hi in PPM_BUCKETS]
    x      = np.arange(len(PPM_BUCKETS))
    bar_w  = 0.8 / len(loaded)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    for i, mode in enumerate(loaded):
        pred = np.array(results[mode]["test_pred"])
        true = np.array(results[mode]["test_true"])
        bucket_maes = []
        for lo, hi in PPM_BUCKETS:
            mask = (true >= lo) & (true < hi)
            bucket_maes.append(np.mean(np.abs(pred[mask] - true[mask])) if mask.sum() > 0 else 0.0)
        offset = (i - (len(loaded) - 1) / 2) * bar_w
        bars = ax.bar(x + offset, bucket_maes, bar_w, label=LABELS[mode],
                      color=COLORS[mode], edgecolor="white", linewidth=0.5)
        for bar, v in zip(bars, bucket_maes):
            if v > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                        f"{v:.1f}", ha="center", va="bottom", fontsize=7)

    ax.set_xticks(x)
    ax.set_xticklabels(bucket_labels)
    ax.set_xlabel("True shift range (ppm)")
    ax.set_ylabel("Mean Absolute Error (ppm)")
    ax.set_title("MAE by Chemical Shift Range (Test Set)", fontweight="bold")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig("fig_mae_by_range.png", bbox_inches="tight")
    plt.show()